<a href="https://colab.research.google.com/github/TarunScript/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarunScript/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is content refresh prioritization. I'm framing this as a scoring task, not classification — instead of a hard yes/no "refresh or don't," I want a continuous score that ranks pages by how urgently they need attention. This lets a content team triage a list rather than get a binary flag with no sense of priority.

In [55]:
import os

if not os.path.exists('/content/flyrank-ml-internship'):
    !git clone https://github.com/TarunScript/flyrank-ml-internship.git

os.chdir('/content/flyrank-ml-internship')
print(os.getcwd())
os.listdir("data/raw")

/content/flyrank-ml-internship


['content_refresh_anonymized.csv']

In [56]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.shape

(30000, 44)

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no direct "needs refresh" label in the data, so I need a proxy. A reasonable proxy: pages with declining traffic/rankings over time relative to their age, since a true "needs refresh" label doesn't exist — I'm inferring it from observed performance decay, not a ground-truth outcome.

In [57]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [58]:
df[['content_id', 'trend_direction', 'trend_pct', 'days_since_last_update', 'freshness_tier']].sample(5)

,content_id,trend_direction,trend_pct,days_since_last_update,freshness_tier
10272,content_95f832767083,up,50.0,20,0-30
20921,content_40e0e5c35df4,up,41.3,22,0-30
17998,content_2021dc8bd93e,stable,17.1,104,91-180
10116,content_987da6440098,down,-94.6,20,0-30
10714,content_8aef9b028f6e,new,NaN,20,0-30


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'd use precision@k — if I rank pages by predicted refresh-priority and take the top 20, what fraction actually show real performance decline? This matters more than raw accuracy because the real use case is "give me a short list to act on," not "score every page perfectly."

In [59]:
df['priority_score'] = (-df['trend_pct']) * (df['days_since_last_update'])
top_20 = df.nlargest(20, 'priority_score')
top_20[['content_id', 'trend_pct', 'days_since_last_update', 'priority_score']]

,content_id,trend_pct,days_since_last_update,priority_score
29384,content_f6fdf87348f6,-100.0,373,37300.0
24216,content_1b4ec72dafd4,-100.0,372,37200.0
26242,content_55a5b1c46474,-88.5,373,33010.5
7509,content_7a888d3d99c8,-100.0,313,31300.0
18841,content_94991fe6268c,-100.0,313,31300.0
22860,content_ab18b5811c02,-100.0,305,30500.0
24557,content_84d12054c0c0,-100.0,304,30400.0
19420,content_ccfb4d0227b1,-100.0,301,30100.0
21984,content_02b0d6e30129,-95.6,313,29922.8
23741,content_df1fa766cac2,-97.7,304,29700.8


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page/URL, observed at one point in time.


In [60]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,priority_score
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,828.0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1442.5
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1218.0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,303.6
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,485.8


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag anything not updated in 12 months" ignores that some evergreen pages stay relevant for years while others decay in weeks. The signal is a messy combination of traffic trend, content age, ranking volatility, and topic — no single threshold captures that interaction. A model can weigh these factors jointly instead of guessing one cutoff.

In [61]:
df[['content_age_days', 'trend_pct', 'freshness_tier']].corr(numeric_only=True)

,content_age_days,trend_pct
content_age_days,1.000000,0.000712
trend_pct,0.000712,1.000000


In [62]:
df = df.drop(columns=['client_id'])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.